# Near-Duplicate Hunting With Cosine Similarity

**Topic:** Norms and cosine similarity &nbsp;·&nbsp; **Difficulty:** Easy

A document index has picked up near-duplicates: the same article re-posted with a
few words changed. You have bag-of-words vectors and need to find every pair
whose cosine similarity is at least 0.95.

Two traps are waiting. Document lengths vary by two orders of magnitude, so raw
dot products rank by length rather than by content. And a handful of documents
are empty, so normalizing them divides by zero.

## The data (created in the next cell)

- `docs` `(800, 60)`: non-negative bag-of-words vectors.
- Some rows are exact scalar multiples of others (the re-posts).
- Exactly 5 rows are all zeros.

## Your task

With **NumPy only**:

1. L2-normalize each row. Leave all-zero rows as all-zero rather than producing
   `NaN` or `inf`.
2. Compute the full `(800, 800)` cosine similarity matrix.
3. Count unordered pairs `i < j` with similarity `>= 0.95`.
4. Set `answers` to a dict with `"n_zero_rows"` (int), `"n_duplicate_pairs"`
   (int), and `"max_offdiagonal"` (float, the largest similarity between two
   different documents).

## Evaluation

Counts must match exactly, the maximum to 2e-6. A `NaN` in any of your
three answer values fails.

In [ ]:
import numpy as np

rng = np.random.default_rng(41)
n, d = 800, 60
docs = rng.poisson(1.2, (n, d)).astype(float)
scale = rng.uniform(1, 120, (n, 1))
docs *= scale
for src, dst in [(10, 700), (11, 701), (12, 702), (13, 703), (14, 704),
                 (15, 705), (16, 706)]:
    docs[dst] = docs[src] * rng.uniform(0.5, 3.0)     # exact re-post
docs[[100, 200, 300, 400, 500]] = 0.0
print(f"docs {docs.shape}, zero rows: {int((docs.sum(axis=1) == 0).sum())}")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("n_zero_rows", [], 5.0, -0.8371900740724958, 1e-06),
    ("n_duplicate_pairs", [], 7.0, -1.172066103701494, 1e-06),
    ("max_offdiagonal", [], 1.0000000000000004, -0.16743801481449924, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

Guard the division with `np.where(norms == 0, 1.0, norms)` so the zero rows stay
zero instead of becoming `NaN`. Their similarity to everything is then 0, which
is the sensible answer. After normalizing, the Gram matrix `U @ U.T` *is* the
cosine similarity matrix, and `np.triu(..., k=1)` picks out the unordered
pairs.

```python
import numpy as np

norms = np.linalg.norm(docs, axis=1, keepdims=True)
safe = np.where(norms == 0, 1.0, norms)
U = docs / safe

S = U @ U.T
iu = np.triu_indices(S.shape[0], k=1)
off = S[iu]

answers = {
    "n_zero_rows": int((norms.ravel() == 0).sum()),
    "n_duplicate_pairs": int((off >= 0.95).sum()),
    "max_offdiagonal": float(off.max()),
}
```